Section 2.4 Content-based recs

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity

from recsys.data.loaders import load_movielens
from recsys.utils.colab import get_data_path


In [ ]:

DATA_PATH = get_data_path()
ratings, movies = load_movielens("ml-25m", data_dir=DATA_PATH)
ratings.head()

In [ ]:
# Keep ratings/movies loaded from load_movielens in Cell 2
print(ratings.head())
print(f"ratings shape: {ratings.shape}")
print(movies.head())
print(f"movies shape: {movies.shape}")

In [ ]:
from scipy.sparse import csr_matrix

user_ids = ratings['userId'].unique()
movie_ids = ratings['movieId'].unique()
user_to_idx = {uid: idx for idx, uid in enumerate(user_ids)} #A
movie_to_idx = {mid: idx for idx, mid in enumerate(movie_ids)} #A
idx_to_movie = {idx: mid for mid, idx in movie_to_idx.items()} #A

rows = [user_to_idx[uid] for uid in ratings['userId']] #B
cols = [movie_to_idx[mid] for mid in ratings['movieId']] #B
data = [1] * len(ratings) #B

user_item_matrix = csr_matrix(
    (data, (rows, cols)),
    shape=(len(user_ids), len(movie_ids))
) #C
print(f"user_item_matrix shape: {user_item_matrix.shape}")

In [ ]:
movie_counts = ratings['movieId'].value_counts() #A
max_count = movie_counts.max() #B

popularity_scores = {
    int(mid): float(count / max_count)
    for mid, count in movie_counts.items()
}


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

item_similarity = cosine_similarity(user_item_matrix.T, dense_output=False) #A
print(f"Similarity matrix shape: {item_similarity.shape}")
print(f"Similarity matrix memory: {item_similarity.data.nbytes / 1024 / 1024:.1f} MB")


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
movies["clean_title"] = (
    movies["title"]
    .str.replace(r"\s*\((?:19|20)\d{2}(?:-(?:19|20)\d{2})?\)\s*$", "", regex=True)
    .str.strip()
)
movies['content'] = movies['clean_title'] + ' ' + movies['genres'].fillna('')

vectorizer = TfidfVectorizer(
    max_features=500,
    stop_words='english',
    token_pattern=r'(?u)\b\w+\b'
)

content_vectors = vectorizer.fit_transform(movies['content'])

print(f"Content vectors shape: {content_vectors.shape}")
print(f"Memory: {content_vectors.data.nbytes / 1024 / 1024:.1f} MB")

print(f"\nTop features (most distinctive words):")
feature_names = vectorizer.get_feature_names_out()

for word in feature_names[:10]:
    print(f"  {word}")

idf_scores = vectorizer.idf_
# Top 10 most distinctive words
top_idx = np.argsort(idf_scores)[-10:][::-1]

print("Top 10 most distinctive words (highest IDF):")
for word, score in zip(feature_names[top_idx], idf_scores[top_idx]):
    print(f"{word}: {score:.4f}")

In [ ]:
movies['content']


In [ ]:
# If you want rows where the content text contains the word "genres"
rows_with_genres_word = movies[movies["content"].str.contains(r"\bgenres\b", case=False, na=False)]
rows_with_genres_word.head()

In [ ]:
movie_id_to_idx = {
    int(mid): idx 
    for idx, mid in enumerate(movies['movieId'])
} #A
idx_to_movie_id = {idx: mid for mid, idx in movie_id_to_idx.items()} #A

def retrieve_similar_by_content(movie_id, k=100):
    if movie_id not in movie_id_to_idx:
        return []
    
    movie_idx = movie_id_to_idx[movie_id] #B
    movie_vector = content_vectors[movie_idx] #B
    
    similarities = cosine_similarity(movie_vector, content_vectors)[0] #C
    
    top_indices = np.argsort(similarities)[-(k+1):-1][::-1] #D
    
    candidates = []
    for idx in top_indices:
        candidates.append({
            'movie_id': int(idx_to_movie_id[idx]),
            'content_similarity': float(similarities[idx])
        })
    
    return candidates


In [ ]:
input_movie_id = 209157
candidates = retrieve_similar_by_content(input_movie_id, k=10)
title = movies.loc[movies['movieId'] == input_movie_id, 'title'].values[0]
print(f"Input movie, ID: {input_movie_id}, Title: {title}")
print("Similar movies based on content:")
for i, item in enumerate(candidates, 1):
    title = movies[movies['movieId'] == item['movie_id']]['title'].values[0]
    score = item['content_similarity']
    genres = movies[movies['movieId'] == item['movie_id']]['genres'].values[0]
    
    print(f"{item['movie_id']}.\t {title}, sim: {score:.4f}, genres: {genres}")


    

In [ ]:
movies[movies['movieId'] == 209157]


In [ ]:
def get_user_history(user_id, k=None) -> int: #A
    user_rows = ratings[ratings["userId"] == user_id]
    user_rows = user_rows.sort_values("timestamp")
    movie_ids = user_rows["movieId"].tolist()
    return set(movie_ids) if k is None else set(movie_ids[-k:])


def filter_watched(candidates, user_id) -> list[dict]: #B
    user_history = get_user_history(user_id)
    
    filtered = [
        item for item in candidates
        if item['movie_id'] not in user_history
    ]    
    return filtered
def score_popularity(candidates):
    for item in candidates:
        item['popularity'] = popularity_scores.get(item['movie_id'], 0.0)
    return candidates


def recommend_content_based(user_id, k=10, content_weight=0.7):
  user_history = get_user_history(user_id)
    
  if len(user_history) == 0:
    popular_movies = movie_counts.head(k).index.tolist()
    return [{'movie_id': int(mid)} for mid in popular_movies]
    
  all_candidates = {}
  recent_movies = list(user_history)[-20:]
    
  for movie_id in recent_movies:
    candidates = retrieve_similar_by_content(movie_id, k=50)
        
    for item in candidates:
      mid = item['movie_id']
      score = item['content_similarity']
      if mid in all_candidates:
        all_candidates[mid] = max(all_candidates[mid], score)
      else:
        all_candidates[mid] = score
    
  candidates = [
    {'movie_id': mid, 'content_similarity': score}
    for mid, score in all_candidates.items()
  ]
  candidates = filter_watched(candidates, user_id)

  candidates = score_popularity(candidates)
      
  for item in candidates:
    item['final_score'] = (
      content_weight * item['content_similarity'] +
      (1 - content_weight) * item['popularity']
      )
    
  ranked = sorted(candidates, key=lambda x: x['final_score'], reverse=True)
    
  return ranked[:k]


In [ ]:
recs = recommend_content_based(user_id=1, k=10, content_weight=0.7)
for i, item in enumerate(recs, 1):
    title = movies[movies['movieId'] == item['movie_id']]['title'].values[0]
    score = item['content_similarity']
    genres = movies[movies['movieId'] == item['movie_id']]['genres'].values[0]
    
    print(f"{item['movie_id']}.\t {title}, sim: {score:.4f}, genres: {genres}")


    

In [ ]:
import importlib

import recsys
importlib.reload(recsys.fourstage_recsys.retrieval.itemknn_retrieval)
importlib.reload(recsys.fourstage_recsys.filtering.history_filtering)
from recsys.fourstage_recsys.retrieval.itemknn_retrieval import ItemKNNRetrieval
from recsys.fourstage_recsys.filtering.history_filtering import HistoryFiltering

item_knn_retrieval = ItemKNNRetrieval(ratings)
history_filtering = HistoryFiltering(ratings)

def retrieve_hybrid(movie_id, k=100):
    content_candidates = retrieve_similar_by_content(movie_id, k=k//2)
    behavioral_candidates = item_knn_retrieval.retrieve_similar_items(movie_id, k=k//2)
    
    all_candidates = {}
    
    for item in content_candidates:
        mid = item['movie_id']
        all_candidates[mid] = {
            'movie_id': mid,
            'content_similarity': item['content_similarity'],
            'behavioral_similarity': 0.0
        }
    
    for item in behavioral_candidates:
        mid = item['movie_id']
        if mid in all_candidates:
            all_candidates[mid]['behavioral_similarity'] = item['similarity']
        else:
            all_candidates[mid] = {
                'movie_id': mid,
                'content_similarity': 0.0,
                'behavioral_similarity': item['similarity']
            }
    
    return list(all_candidates.values())

retrieve_hybrid(1, k=10)

In [ ]:

def rank_three_signals(candidates, content_weight=0.3, behavioral_weight=0.5, k=10):
    popularity_weight = 1.0 - content_weight - behavioral_weight
    
    for item in candidates:
        item['final_score'] = (
            content_weight * item.get('content_similarity', 0) +
            behavioral_weight * item.get('behavioral_similarity', 0) +
            popularity_weight * item.get('popularity', 0)
        )
    
    ranked = sorted(candidates, key=lambda x: x['final_score'], reverse=True)
    return ranked[:k]

seed_movie_id = 1
candidates = retrieve_hybrid(seed_movie_id, k=100)
candidates = add_popularity_scores(candidates)

ranked = rank_three_signals(
    candidates,
    content_weight=0.3,
    behavioral_weight=0.5
)

print("Hybrid recommendations (content + behavior + popularity):\n")
for i, item in enumerate(ranked[:5], 1):
    title = movies[movies['movieId'] == item['movie_id']]['title'].values[0]
    print(f"{i}. {title}")
    print(f"   Content: {item.get('content_similarity', 0):.3f}")
    print(f"   Behavior: {item.get('behavioral_similarity', 0):.3f}")
    print(f"   Popularity: {item.get('popularity', 0):.3f}")
    print(f"   Final: {item['final_score']:.3f}\n")


In [ ]:
from recsys import FourStageRecommender #A
from recsys.retrievals import ItemKNNRetrieval #B
from recsys.filters import HistoryFilter #B
from recsys.scorers import PopularityScorer #B
from recsys.rankers import WeightedRanker #B

retrieval = ItemKNNRetrieval(
    interactions_path='data/ratings.csv',
    items_path='data/movies.csv'
) #C

recommender = FourStageRecommender(
  retrieval=retrieval,
  filter=HistoryFilter(),          
  scorer=PopularityScorer(interactions_path='data/ratings.csv'),
  ranker=WeightedRanker(weights={'similarity': 0.7, 'popularity': 0.3})
) #D

recommendations = recommender.recommend(
    user_id='123',
    k=10
) #H
